# RoomFormer ONNX Export — Step 3 (v2)

This notebook exports RoomFormer to ONNX on a **free Colab T4 GPU**.

**Key change from v1**: We skip building the CUDA extension entirely.
Instead, we stub out the CUDA import and patch in the pure-PyTorch
attention *before* any RoomFormer code loads. This avoids the
PyTorch 1.9 / CUDA 11.1 incompatibility with Colab's environment.

**Trade-off**: We skip PATCH.2 (comparing patched vs original output)
since we can't run the original CUDA model. The pure-PyTorch fallback
uses the same math (`F.grid_sample` bilinear interpolation) so the
outputs are mathematically equivalent.

**Instructions**: Runtime → Change runtime type → **T4 GPU** → Run All

**Runtime**: ~10 minutes

## 0. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU — will export on CPU (slower but works)")

## 1. Clone RoomFormer + Install Dependencies

We install detectron2 from the official pre-built wheels (not the vendored copy).

In [ ]:
%%bash
cd /content
if [ ! -d "RoomFormer" ]; then
    git clone https://github.com/ywyue/RoomFormer.git
    echo "Cloned RoomFormer"
else
    echo "RoomFormer already exists"
fi

In [ ]:
%%bash
# Install detectron2 from official pre-built wheels for Colab's PyTorch
pip install -q shapely descartes

# Try official detectron2 install
python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git' 2>&1 | tail -3
echo "Dependencies installed"

## 2. Stub Out CUDA Extension + Patch Attention

We intercept the CUDA extension import and replace it with the
pure-PyTorch implementation BEFORE any RoomFormer code tries to load it.

In [ ]:
import sys
import types
import importlib

# Add RoomFormer to path
sys.path.insert(0, '/content/RoomFormer')

# --- Stub ALL compiled C extensions BEFORE any RoomFormer imports ---

# 1. MultiScaleDeformableAttention (models/ops/ — deformable attention CUDA kernel)
fake_cuda_module = types.ModuleType('MultiScaleDeformableAttention')
def _stub(*args, **kwargs):
    raise RuntimeError("Stub called — should have been patched out")
fake_cuda_module.ms_deform_attn_forward = _stub
fake_cuda_module.ms_deform_attn_backward = _stub
sys.modules['MultiScaleDeformableAttention'] = fake_cuda_module
print("[STUB] MultiScaleDeformableAttention")

# 2. native_rasterizer (diff_ras/polygon.py — differentiable rasterization, training only)
fake_rasterizer = types.ModuleType('native_rasterizer')
fake_rasterizer.rasterize_forward = _stub
fake_rasterizer.rasterize_backward = _stub
# Add common attribute patterns the module might use
for attr in ['forward', 'backward', 'rasterize', 'RasterizeFunction']:
    setattr(fake_rasterizer, attr, _stub)
sys.modules['native_rasterizer'] = fake_rasterizer
print("[STUB] native_rasterizer")

# --- Now safe to import RoomFormer modules ---
import torch
import torch.nn.functional as F
import numpy as np

from models.ops.functions.ms_deform_attn_func import ms_deform_attn_core_pytorch
from models.ops.modules import ms_deform_attn as attn_module

# --- Monkey-patch deformable attention to pure-PyTorch ---
def _patched_forward(self, query, reference_points, input_flatten,
                     input_spatial_shapes, input_level_start_index,
                     input_padding_mask=None):
    """Pure-PyTorch forward using F.grid_sample instead of CUDA kernel."""
    N, Len_q, _ = query.shape
    N, Len_in, _ = input_flatten.shape

    value = self.value_proj(input_flatten)
    if input_padding_mask is not None:
        value = value.masked_fill(input_padding_mask[..., None], float(0))
    value = value.view(N, Len_in, self.n_heads, self.d_model // self.n_heads)

    sampling_offsets = self.sampling_offsets(query).view(
        N, Len_q, self.n_heads, self.n_levels, self.n_points, 2)
    attention_weights = self.attention_weights(query).view(
        N, Len_q, self.n_heads, self.n_levels * self.n_points)
    attention_weights = F.softmax(attention_weights, -1).view(
        N, Len_q, self.n_heads, self.n_levels, self.n_points)

    if reference_points.shape[-1] == 2:
        offset_normalizer = torch.stack(
            [input_spatial_shapes[..., 1], input_spatial_shapes[..., 0]], -1)
        sampling_locations = (
            reference_points[:, :, None, :, None, :]
            + sampling_offsets / offset_normalizer[None, None, None, :, None, :]
        )
    elif reference_points.shape[-1] == 4:
        sampling_locations = (
            reference_points[:, :, None, :, None, :2]
            + sampling_offsets / self.n_points
            * reference_points[:, :, None, :, None, 2:]
            * 0.5
        )
    else:
        raise ValueError(f"reference_points last dim must be 2 or 4, got {reference_points.shape[-1]}")

    output = ms_deform_attn_core_pytorch(
        value, input_spatial_shapes, sampling_locations, attention_weights)

    output = self.output_proj(output)
    return output

attn_module.MSDeformAttn.forward = _patched_forward
print("[PATCH] MSDeformAttn.forward() → pure-PyTorch (F.grid_sample)")
print("\nAll stubs and patches applied — ready to build model")

## 3. Download Pretrained Checkpoint

In [ ]:
%%bash
cd /content/RoomFormer

# Remove corrupted file if it exists but is too small (HTML redirect page)
if [ -f "checkpoint_s3d.pth" ]; then
    SIZE=$(stat -c%s "checkpoint_s3d.pth" 2>/dev/null || stat -f%z "checkpoint_s3d.pth")
    if [ "$SIZE" -lt 1000000 ]; then
        echo "Existing file is too small (${SIZE} bytes) — likely corrupted. Re-downloading..."
        rm checkpoint_s3d.pth
    fi
fi

if [ ! -f "checkpoint_s3d.pth" ]; then
    echo "Downloading Structured3D checkpoint..."
    echo "Trying Polybox with curl -L (follows redirects)..."
    curl -L -o checkpoint_s3d.pth \
        "https://polybox.ethz.ch/index.php/s/vlBo66X0NTrcsTC/download" \
        2>&1 | tail -3

    # Check if we got a real checkpoint or an HTML page
    SIZE=$(stat -c%s "checkpoint_s3d.pth" 2>/dev/null || stat -f%z "checkpoint_s3d.pth")
    if [ "$SIZE" -lt 1000000 ]; then
        echo "WARNING: File too small (${SIZE} bytes). Polybox link may be broken."
        echo "Checking if it's HTML..."
        head -c 100 checkpoint_s3d.pth
        echo ""
        rm checkpoint_s3d.pth

        echo ""
        echo "Trying alternative: RoomFormer Google Drive link..."
        pip install -q gdown 2>/dev/null
        # Try known Google Drive alternatives for RoomFormer checkpoints
        # Check the repo README for current links
        python3 -c "
import subprocess, re, urllib.request

# Fetch README to find checkpoint links
url = 'https://raw.githubusercontent.com/ywyue/RoomFormer/main/README.md'
readme = urllib.request.urlopen(url).read().decode()

# Look for download links
links = re.findall(r'https?://[^\s\)]+(?:download|\.pth|drive\.google)[^\s\)]*', readme)
print('Found links in README:')
for l in links:
    print(f'  {l}')
"
    fi
fi

if [ -f "checkpoint_s3d.pth" ]; then
    ls -lh checkpoint_s3d.pth
    echo "Checkpoint ready"
else
    echo ""
    echo "=========================================="
    echo "MANUAL DOWNLOAD NEEDED"
    echo "=========================================="
    echo "1. Go to the RoomFormer repo: https://github.com/ywyue/RoomFormer"
    echo "2. Find the Structured3D checkpoint link in the README"
    echo "3. Download it manually and upload to /content/RoomFormer/checkpoint_s3d.pth"
    echo "   (Use the Files panel on the left → Upload)"
    echo ""
    echo "Or run this in the next cell:"
    echo "  from google.colab import files"
    echo "  uploaded = files.upload()  # select your .pth file"
    echo "  import shutil"
    echo "  shutil.move(list(uploaded.keys())[0], '/content/RoomFormer/checkpoint_s3d.pth')"
fi

In [ ]:
# --- Run this cell ONLY if the auto-download above failed ---
# It will open a file picker to upload the checkpoint manually.
import os
if not os.path.exists('/content/RoomFormer/checkpoint_s3d.pth'):
    print("Auto-download failed. Upload the checkpoint manually.")
    print("Get it from the RoomFormer README: https://github.com/ywyue/RoomFormer")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        import shutil
        name = list(uploaded.keys())[0]
        shutil.move(name, '/content/RoomFormer/checkpoint_s3d.pth')
        print(f"Moved {name} → /content/RoomFormer/checkpoint_s3d.pth")
else:
    print("Checkpoint exists — skip this cell")

## 4. Build Model + Test Forward Pass (PATCH.1)

In [ ]:
from models import build_model
from main import get_args_parser
from util.misc import NestedTensor

# Build model
parser = get_args_parser()
args = parser.parse_args([
    '--num_queries', '800',
    '--num_polys', '20',
    '--num_feature_levels', '4',
    '--backbone', 'resnet50',
    '--with_poly_refine',
    '--masked_attn',
    '--semantic_classes', '-1',
])

model = build_model(args, train=False)

# Load checkpoint
# weights_only=False is needed because PyTorch 2.x defaults to safe-mode
# which can't load old PyTorch 1.x pickle-format checkpoints
ckpt_path = '/content/RoomFormer/checkpoint_s3d.pth'
print(f"Loading checkpoint ({os.path.getsize(ckpt_path)/1e6:.1f} MB)...")
try:
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
except TypeError:
    # PyTorch < 2.0 doesn't have weights_only param
    ckpt = torch.load(ckpt_path, map_location='cpu')

missing, unexpected = model.load_state_dict(ckpt['model'], strict=False)
print(f"Loaded: {len(missing)} missing, {len(unexpected)} unexpected keys")
if missing:
    print(f"  Missing (first 5): {missing[:5]}")
if unexpected:
    print(f"  Unexpected (first 5): {unexpected[:5]}")

model.eval()

# --- PATCH.1: Forward pass on CPU ---
print("\n[PATCH.1] Testing forward pass on CPU...")
model.cpu()
dummy = torch.randn(1, 1, 256, 256)
mask = torch.zeros(1, 256, 256, dtype=torch.bool)
nested = NestedTensor(dummy, mask)

with torch.no_grad():
    out = model(nested)

print(f"  pred_logits: {out['pred_logits'].shape}")  # expect [1, 20, 40]
print(f"  pred_coords: {out['pred_coords'].shape}")  # expect [1, 20, 40, 2]
print(f"  logits range: [{out['pred_logits'].min():.3f}, {out['pred_logits'].max():.3f}]")
print(f"  coords range: [{out['pred_coords'].min():.3f}, {out['pred_coords'].max():.3f}]")
print("[PATCH.1] PASSED — patched model runs on CPU")

# Determinism check
print("\nChecking determinism...")
r1 = model(NestedTensor(dummy, mask))['pred_coords']
r2 = model(NestedTensor(dummy, mask))['pred_coords']
print(f"Determinism: {'PASSED' if torch.equal(r1, r2) else 'WARNING'}")

## 5. Export to ONNX

In [ ]:
import torch.nn as nn
import os

# Rebuild model WITHOUT aux_loss for cleaner ONNX graph
args.aux_loss = False
model_export = build_model(args, train=False)
model_export.load_state_dict(ckpt['model'], strict=False)
model_export.eval()
model_export.cpu()

class RoomFormerONNX(nn.Module):
    """Wrapper that creates NestedTensor internally for ONNX export."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, density_map):
        # density_map: [1, 1, 256, 256]
        B, C, H, W = density_map.shape
        mask = torch.zeros(B, H, W, dtype=torch.bool, device=density_map.device)
        samples = NestedTensor(density_map, mask)
        outputs = self.model(samples)
        return outputs['pred_logits'], outputs['pred_coords']

wrapper = RoomFormerONNX(model_export)
wrapper.eval()

# Verify wrapper works
with torch.no_grad():
    pt_logits, pt_coords = wrapper(dummy)
print(f"Wrapper forward: logits {pt_logits.shape}, coords {pt_coords.shape}")

In [ ]:
ONNX_PATH = '/content/roomformer_s3d.onnx'
EXPORT_SUCCESS = False

print("Exporting to ONNX (opset 16)...")
print("This may take 1-2 minutes...\n")
try:
    torch.onnx.export(
        wrapper,
        (dummy,),
        ONNX_PATH,
        opset_version=16,
        input_names=["density_map"],
        output_names=["pred_logits", "pred_coords"],
        dynamic_axes=None,
        do_constant_folding=True,
        verbose=False,
    )
    size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    print(f"[ONNX.1] PASSED — {ONNX_PATH} ({size_mb:.1f} MB)")
    EXPORT_SUCCESS = True
except Exception as e:
    print(f"[ONNX.1] FAILED — {e}\n")
    print("Attempting TorchScript fallback...")
    TS_PATH = '/content/roomformer_s3d.pt'
    try:
        traced = torch.jit.trace(wrapper, (dummy,))
        traced.save(TS_PATH)
        size_mb = os.path.getsize(TS_PATH) / (1024 * 1024)
        print(f"TorchScript PASSED — {TS_PATH} ({size_mb:.1f} MB)")
        print("\n⚠️  ONNX failed, TorchScript succeeded → Path B")
    except Exception as e2:
        print(f"TorchScript also FAILED — {e2}")
        SD_PATH = '/content/roomformer_patched.pth'
        torch.save({'model': model_export.state_dict(), 'args': vars(args)}, SD_PATH)
        print(f"\nSaved raw state_dict to {SD_PATH} → Path B (TorchServe)")

## 6. Validate ONNX (ONNX.2-5)

In [ ]:
if EXPORT_SUCCESS:
    !pip install -q onnx onnxruntime
    import onnx
    import onnxruntime as ort
    import time

    # ONNX.1 structural check
    onnx_model = onnx.load(ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    print("[ONNX.1] Structural check PASSED")

    # ONNX.2 input/output shapes
    session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
    for inp in session.get_inputs():
        print(f"[ONNX.2] Input:  {inp.name} {inp.shape} {inp.type}")
    for out in session.get_outputs():
        print(f"[ONNX.2] Output: {out.name} {out.shape} {out.type}")

    result = session.run(None, {"density_map": dummy.numpy()})
    onnx_logits, onnx_coords = result[0], result[1]
    print(f"[ONNX.2] Inference PASSED — logits {onnx_logits.shape}, coords {onnx_coords.shape}")

    # ONNX.3 latency
    times = []
    for _ in range(5):
        t0 = time.time()
        session.run(None, {"density_map": dummy.numpy()})
        times.append(time.time() - t0)
    avg = np.mean(times)
    print(f"[ONNX.3] Latency: {avg:.3f}s avg — {'PASSED' if avg < 5.0 else 'FAILED'} (limit 5s)")

    # ONNX.4 determinism
    r1 = session.run(None, {"density_map": dummy.numpy()})
    r2 = session.run(None, {"density_map": dummy.numpy()})
    det = np.allclose(r1[0], r2[0]) and np.allclose(r1[1], r2[1])
    print(f"[ONNX.4] Determinism: {'PASSED' if det else 'FAILED'}")

    # ONNX.5 parity with PyTorch
    ld = np.abs(onnx_logits - pt_logits.numpy()).max()
    cd = np.abs(onnx_coords - pt_coords.numpy()).max()
    print(f"[ONNX.5] PyTorch↔ONNX: logits diff={ld:.6f}, coords diff={cd:.6f}")
    if ld < 1e-3 and cd < 1e-3:
        print("[ONNX.5] PASSED (atol=1e-3)")
    elif ld < 0.01 and cd < 0.01:
        print("[ONNX.5] ACCEPTABLE (atol=0.01)")
    else:
        print("[ONNX.5] WARNING — large difference")

    print("\n" + "="*50)
    print("ALL ONNX TESTS COMPLETE — Path A confirmed")
    print("="*50)
else:
    print("ONNX validation skipped — export did not succeed.")
    print("Check the cell above for TorchScript or state_dict fallback.")

## 7. Download Model

Place the downloaded file at:
```
cloud/processor/models/roomformer_s3d.onnx   (Path A)
cloud/processor/models/roomformer_s3d.pt      (if TorchScript)
cloud/processor/models/roomformer_patched.pth  (if Path B)
```

In [ ]:
from google.colab import files

for path in ['/content/roomformer_s3d.onnx',
             '/content/roomformer_s3d.pt',
             '/content/roomformer_patched.pth']:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"Downloading {path} ({size_mb:.1f} MB)...")
        files.download(path)
        break
else:
    print("No model file found!")

## Results

| Test | Expected |
|------|----------|
| PATCH.1 | CPU forward pass works |
| PATCH.2 | *Skipped* (no CUDA extension to compare against) |
| ONNX.1 | Valid ONNX file |
| ONNX.2 | Input `[1,1,256,256]` → logits `[1,20,40]` + coords `[1,20,40,2]` |
| ONNX.3 | Inference < 5s |
| ONNX.4 | Deterministic |
| ONNX.5 | PyTorch↔ONNX parity < 0.01 |